# Wound Ontology — Term Mapping (Google Colab Pro)
**Runtime → Change runtime type → GPU (T4 or A100)**

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — change runtime type to GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Clone repo & install
The concept store in `data/cache/` comes with the repo — no Drive upload needed.

In [ ]:
import subprocess, sys, os

# Load GitHub token from Colab Secrets (key icon in left sidebar → add GITHUB_TOKEN)
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

assert GITHUB_TOKEN and GITHUB_TOKEN.startswith("ghp_"), \
    "GITHUB_TOKEN missing or wrong format — add it in Colab Secrets (key icon)"

clone_url = f"https://{GITHUB_TOKEN}@github.com/Strubbelig/Wound-ontology.git"
subprocess.run(["git", "clone", clone_url], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", "Wound-ontology", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install",
    "nest_asyncio", "python-dotenv", "-q"], check=True)

print("Installation complete.")

## Mount Google Drive
Drive is only used to save the output mapping. Change `DRIVE_CACHE` if your folder path differs.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_CACHE = "/content/drive/MyDrive/wound_ontology/cache"
os.makedirs(DRIVE_CACHE, exist_ok=True)
print(f"Drive output folder: {DRIVE_CACHE}")

## Set up pipeline

In [ ]:
import sys, nest_asyncio
nest_asyncio.apply()

sys.path.insert(0, "/content/Wound-ontology")
sys.path.insert(0, "/content/Wound-ontology/config")

from pathlib import Path
from wound_ontology.orchestrator import Orchestrator
from wound_ontology.embedding.embedder import Embedder

REPO_CACHE = Path("/content/Wound-ontology/data/cache")

pipeline = Orchestrator(
    base_path=str(REPO_CACHE),
    allowed_predicates=[
        "is_a", "part_of", "has_quality", "occurs_in", "caused_by", "is_treatment_for",
    ],
)
print(f"Concept store loaded: {len(pipeline.store.dict)} PubMed entries")
print(f"Existing mappings   : {len(pipeline.term_mapper.mapping)} terms already mapped")

embedder = Embedder(batch_size=256)

## Run term mapping

Merge condition (both must hold):
- **cosine ≥ `SIMILARITY_THRESHOLD`** (0.93–0.98)
- **string similarity ≥ `STRING_THRESHOLD`** after normalization (case, hyphens, parens stripped)

Exceptions: abbreviation pairs (e.g. DFU ↔ Diabetic Foot Ulcer) bypass string check.  
Generic single-word terms (wound, pain, skin…) require string similarity ≥ 0.90.  
If any cluster reaches ≥ 20 members an error is raised — increase `SIMILARITY_THRESHOLD`.

In [ ]:
SIMILARITY_THRESHOLD = 0.95
STRING_THRESHOLD     = 0.92

freq  = pipeline._collect_store_terms_with_freq()
terms = list(freq.keys())
print(f"Unique terms in store: {len(terms)}")

result = pipeline.term_mapper.build_from_embeddings(
    terms,
    embedder=embedder,
    similarity_threshold=SIMILARITY_THRESHOLD,
    string_threshold=STRING_THRESHOLD,
    frequencies=freq,
)
print(f"Mapping complete. Total entries: {len(result)}")

## Save mapping to Drive

In [ ]:
import glob, shutil

latest_mapping = sorted(glob.glob(str(REPO_CACHE / "term_mapping_*.json")))
if latest_mapping:
    src  = latest_mapping[-1]
    dest = Path(DRIVE_CACHE) / Path(src).name
    shutil.copy(src, dest)
    print(f"Saved to Drive: {Path(dest).name}")
else:
    print("No mapping file found — check for errors above.")